# SECOND NOTEBOOK FOR STUDENT SCORE PREDICTION

In [11]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv('./student1.csv')

In [3]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [5]:
df = df.drop(['race/ethnicity'], axis=1)
df

,gender,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,bachelor's degree,standard,none,72,72,74
1,female,some college,standard,completed,69,90,88
2,female,master's degree,standard,none,90,95,93
3,male,associate's degree,free/reduced,none,47,57,44
4,male,some college,standard,none,76,78,75
...,...,...,...,...,...,...,...
995,female,master's degree,standard,completed,88,99,95
996,male,high school,free/reduced,none,62,55,55
997,female,high school,free/reduced,completed,59,71,65
998,female,some college,standard,completed,68,78,77


### PREPARE DATA

In [7]:
X = df.drop(columns=[
    'math score',
    'reading score',
    'writing score'
])

y_math = df['math score']
y_reading = df['reading score']
y_writing = df['writing score']

In [17]:
X_train, X_test, y_math_train, y_math_test = train_test_split(
    X, y_math,
    test_size=0.2,
    random_state=42
)

_, _, y_reading_train, y_reading_test = train_test_split(
    X, y_reading,
    test_size=0.2,
    random_state=42
)

_, _, y_writing_train, y_writing_test = train_test_split(
    X, y_writing,
    test_size=0.2,
    random_state=42
)

### ENCODE CATEGORICAL COLUMNS

In [9]:
categorical_columns = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            'encoder',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_columns
        )
    ]
)

In [10]:
param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4]
}

In [19]:
def train_model(X_train, X_test, y_train, y_test, score_name):

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor())
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    predictions = best_model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)

    print(f'\n--- {score_name} Model Results ---')
    print('Best Parameters:', grid_search.best_params_)
    print(f'Best Cross Validation MAE: {-grid_search.best_score_:.2f}')
    print(f'Test MAE: {mae:.2f}')
    print(f'Test RMSE: {mse ** 0.5:.2f}')

    return best_model, grid_search

In [20]:
math_model, math_search = train_model(X_train, X_test, y_math_train, y_math_test, 'Math')
reading_model, reading_search = train_model(X_train, X_test, y_reading_train, y_reading_test, 'Reading')
writing_model, writing_search = train_model(X_train, X_test, y_writing_train, y_writing_test, 'Writing')

Fitting 5 folds for each of 81 candidates, totalling 405 fits



--- Math Model Results ---
Best Parameters: {'regressor__max_depth': None, 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 100}
Best Cross Validation MAE: 10.99
Test MAE: 11.54
Test RMSE: 14.56
Fitting 5 folds for each of 81 candidates, totalling 405 fits

--- Reading Model Results ---
Best Parameters: {'regressor__max_depth': 20, 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 100}
Best Cross Validation MAE: 10.62
Test MAE: 11.11
Test RMSE: 14.08
Fitting 5 folds for each of 81 candidates, totalling 405 fits

--- Writing Model Results ---
Best Parameters: {'regressor__max_depth': None, 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 100}
Best Cross Validation MAE: 10.33
Test MAE: 10.53
Test RMSE: 13.69


In [22]:
maths_predictions = math_model.predict(X_test)

results = pd.DataFrame({
    'Actual Math Score': y_math_test.values, 'Predicted Math Score': maths_predictions.round(2)
})

results.head(50)

,Actual Math Score,Predicted Math Score
0,91,66.67
1,53,59.79
2,80,70.54
3,74,61.13
4,84,80.69
5,81,60.71
6,69,69.79
7,54,68.36
8,87,60.71
9,51,72.96
